## XGBoost Churn Prediction

Implements a simple XGBoost model to do customer churn prediction and computes basic evaluations on [Scikit-Learn's Churn Prediction Dataset](https://huggingface.co/datasets/scikit-learn/churn-prediction).

### Step 0: Initalize Enviornment
First ensure the environment is stet up properly with [uv](https://docs.astral.sh/uv/getting-started/installation/).

In [1]:
!uv sync

Resolved 58 packages in 3ms
Checked 52 packages in 1ms


### Step 1: Download the Dataset & Preprocess
The dataset is hosted on Hugging Face, and we can fetch it directly, and store it in a pandas dataframe like so.

In [2]:
import pandas as pd

df = pd.read_csv("hf://datasets/scikit-learn/churn-prediction/dataset.csv")

/Users/dzurec/ai-projects/xgboost-churn-demo/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Now we can inspect the head of the dataset to get an idea of what we're looking at.

In [3]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In order to prepare our dataset for ingestion into XGBoost we need to one-hot encode the data (create dummy variables for categorical variables).

We can create the one-hot encodings like so.

In [4]:
cat_vars = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
]

df = pd.get_dummies(df, columns=cat_vars, drop_first=True)

# encode the target variable as 0 (not going to churn) and 1 (going to churn)
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

# uncomment to see the first few rows of the dataframe
# df.head()

Unfortunately, there's one more problem with our dataset! There are rows where total charge is missing!

In [5]:
print(len(df[df["TotalCharges"] == " "]))

11


Since we have no way to really know the total charges for these toy customers, and for the sake of the demo we will simply fill these rows with the average of the TotalCharges column.

In [6]:
total_charges_mean = df[df["TotalCharges"] != " "]["TotalCharges"].astype(float).mean()

df["TotalCharges"] = df["TotalCharges"].replace(" ", total_charges_mean).astype(float)

Now there should be no more empty total charge samples.

In [7]:
print(len(df[df["TotalCharges"] == " "]))

0


### Step 2: Split the data
We can simply split our tabular data into input/output, and test/validation.

NOTE: In production K-Fold analysis and comprehensive grid-search should be used to find the best model parameters, but for simplicity we skip this step.

Since each sample or row of our data is independent we first shuffle the dataset.

In [8]:
import sklearn

# set random state for reproducibility
RANDOM_STATE = 42

shuffled_df = sklearn.utils.shuffle(df, random_state=RANDOM_STATE)

Now we split our data into features (X) and targets (y). Our features are every column of the table (excluding `customerID`) and our target is `Churn`.

NOTE: We drop the `customerID` because decision trees are greedy and leaving `customerID` may lead to an over fitting scenario where the model effectively predicts solely off the `customerID`!

In [9]:
# First drop the customerID col
shuffled_df = shuffled_df.drop(columns=["customerID"])

# Separate the target variable from the features

y = shuffled_df["Churn"]
X = shuffled_df.drop(columns=["Churn"])

Now we can create the test and validation data, we'll reserve 20% of our data for validation, that is to say we will purposefully "hide" this data from the model so we can better understand how the model will make predictions on data it has not seen.

In [10]:
X_train, X_val, y_train, y_val = sklearn.model_selection.train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

### Step 3: Train the XGBoost Classifier
Using Scikit-Learn we can train the model using standard hyperparameter settings like so.

In [11]:
import xgboost as xgb

# define the model with hyperparameters
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1,
    eval_metric="logloss",
    early_stopping_rounds=20, # NOTE: this monitors eval loss and stops training if it doesn't improve for 20 rounds
    random_state=RANDOM_STATE,
)

# Train the model!
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=True)

[0]	validation_0-logloss:0.54875
[1]	validation_0-logloss:0.53041
[2]	validation_0-logloss:0.51510
[3]	validation_0-logloss:0.50260
[4]	validation_0-logloss:0.49416
[5]	validation_0-logloss:0.48426
[6]	validation_0-logloss:0.47623
[7]	validation_0-logloss:0.47010
[8]	validation_0-logloss:0.46436
[9]	validation_0-logloss:0.45888
[10]	validation_0-logloss:0.45473
[11]	validation_0-logloss:0.45064
[12]	validation_0-logloss:0.44783
[13]	validation_0-logloss:0.44465
[14]	validation_0-logloss:0.44196
[15]	validation_0-logloss:0.43988
[16]	validation_0-logloss:0.43738
[17]	validation_0-logloss:0.43538
[18]	validation_0-logloss:0.43339
[19]	validation_0-logloss:0.43152
[20]	validation_0-logloss:0.42932
[21]	validation_0-logloss:0.42851
[22]	validation_0-logloss:0.42791
[23]	validation_0-logloss:0.42699
[24]	validation_0-logloss:0.42628
[25]	validation_0-logloss:0.42546
[26]	validation_0-logloss:0.42448
[27]	validation_0-logloss:0.42335
[28]	validation_0-logloss:0.42289
[29]	validation_0-loglos

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",20
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


### Step 4: Evaluate the Model
Now we can do some simple evaluations.

First, we can use our new model to make predictions for a future customer so take the first customer from our validation set for example.

In [18]:
future_customer = X_val.iloc[[0]]

print(future_customer)

      tenure  MonthlyCharges  TotalCharges  gender_Male  SeniorCitizen_1  \
3862      70            25.4       1782.05         True            False   

      Partner_Yes  Dependents_Yes  PhoneService_Yes  \
3862         True            True              True   

      MultipleLines_No phone service  MultipleLines_Yes  ...  \
3862                           False               True  ...   

      StreamingTV_No internet service  StreamingTV_Yes  \
3862                             True            False   

      StreamingMovies_No internet service  StreamingMovies_Yes  \
3862                                 True                False   

      Contract_One year  Contract_Two year  PaperlessBilling_Yes  \
3862              False               True                 False   

      PaymentMethod_Credit card (automatic)  PaymentMethod_Electronic check  \
3862                                  False                           False   

      PaymentMethod_Mailed check  
3862                      

And now we can use our trained XGBoost Classifier to predict whether they are likely to churn or not.

In [23]:
model.predict_proba(future_customer)[0, 1]

np.float32(0.010889958)

We can see the model predicts the probability this customer churns is ~1%.

#### Accuracy
Evaluate accuracy on the validation dataset.

In [24]:
accuracy = sklearn.metrics.accuracy_score(y_val, model.predict(X_val))
print(f"Accuracy: {accuracy}")

Accuracy: 0.794180269694819


#### Precision
Evaluate precision on the validation dataset.

In [25]:
precision = sklearn.metrics.precision_score(y_val, model.predict(X_val))
print(f"Precision: {precision}")

Precision: 0.6270627062706271


#### Recall
Evaluate recall on the validation dataset.

In [26]:
recall = sklearn.metrics.recall_score(y_val, model.predict(X_val))
print(f"Recall: {recall}")

Recall: 0.5177111716621253


F1-Score

Evaluate F1-Score on the validation dataset.

In [27]:
f1 = sklearn.metrics.f1_score(y_val, model.predict(X_val))
print(f"F1 Score: {f1}")

F1 Score: 0.5671641791044776
